# Tokenization

Building simple tokenizers from scratch to get practice coding the text -> tokens -> token IDs process, then implementing BPE.

Corpus derived from my all time favorite song: 
*II. Earth: The Oldest Computer (The Last Night)* by *Donald Glover (Childish Gambino)*
https://www.youtube.com/watch?v=6IMmuOyFjQ8

In [ ]:
corpus = (
# [Chorus: Childish Gambino]
"""
See now, I don't wanna see an era, an era, an era
See now, I just wanna live forever, and ever
Maybe it's the last night
Maybe it's the last night
"""

# [Verse 1: Childish Gambino]
"""
When the world was discerned with this and that
I was young, didn't know to hold it back
Now we here and the world is something else
We could leave any day and call for help
We were gods, nobody was above me (Woo)
That "A" on my chest like adultery (Yeah)
That "A" on my chest, put your fist up (Yeah)
That "A" on my chest like a chipmunk
Alvin, Theodore
Breathe in, breathe out, mi amor
Breathe in, breathe out, never mind
It was time, number nine
L-L-Lies, feline with a death wish
Eatin' right, didn't help shit
No name on the guest list
Hold on, we were destined
"""

# [Chorus: Azealia Banks & Childish Gambino]
"""
See now, I don't wanna see an era (No) An era (No) an era
See now, I just wanna live forever and ever
Maybe it’s the last night (You and me)
Maybe it’s the last night (Fireworks)
Maybe it’s the last night (Oh, oh, all the stars)
Maybe it’s the last night (Supermoon)
Maybe it's the last night
"""

# [Bridge: Azealia Banks]
"""
All that I have
All I don't have's the future
Take on my soul
All night
"""

# [Verse 2: Childish Gambino]
"""
Yeah, break free from all of the insides
The godless denied us
But we don't give a damn 'bout the next day
We were never here, never know if the world change
Common sense, the consequences
Retweet the truth, then regret the mentions
We are the wireless survivors of things gone
Slaves to the unnamed, never live long
Live and die by the line, "Never know fear"
Even I won't survive, is it unfair?
Is it unfair? 'Cause I don't care
When I step on that ant on the grass
Progress only thing that'll last (Last, last)
3005, the year that we fear only God will survive
To be alone is alive
"""

# [Chorus: Childish Gambino, Azealia Banks & Both]
"""
See now, I don’t wanna see an era, an era, an era
See now, I just wanna live forever and ever
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
See now, I don’t wanna see an era, an era, an era
See now, I just wanna live forever and ever
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
Maybe it’s the last night
"""

# [Breakdown: Childish Gambino]
"""
She got that body, oh, me, oh, my
Send them pics to my phone, GPOY
She got that body, oh, me, oh, my
Send them pics to my phone, GPOY
GPOY, GPOY
GPOY, GPOY
GPOY, GPOY
Send them pics to my phone, GPOY
"""

# [Outro: Childish Gambino]
"""
So fly
Girl, you know you're so fly, so fly
Girl, I know I miss you
Girl, you know that I miss you
So fly
Girl, you know you're so fly, so fly
Girl, I know I miss you (I do)
Girl, you know that I miss you (I do, ooh)
You're so fly
Girl, you know you're so fly, so fly (I do, I do)
Girl, I know I miss you (Ah, I do)
Girl, you know that I miss you (I do, I do)
You're so fly
Girl, you know you're so fly, so fly (Ooh, ooh)
Girl, I know I miss you (Ooh)
Girl, you know that I miss you (Ooh, oh)
"""
# [Produced by Childish Gambino, Ludwig Göransson & Stefan Ponce]
)

print(corpus)

## Character-Level Tokenization

We implement character-level tokenization first as in terms of code, it's the most intuitive.

In [ ]:
# Get every unique character in training corpus
characters = sorted(set(corpus))

print(characters)
print("Vocabulary size:", len(characters)) # print number of unique characters

We then map each unique character to a token ID, to keep it simple we'll just use `enumerate()`.

In [ ]:
# Map each character to a unique token ID
char_to_id = {
    char: index # character:token ID pair
    for index, char in enumerate(characters)
}

print(char_to_id)

# Reverse vocabulary (decoding purposes)
id_to_char = {
    index: char
    for char, index in char_to_id.items()
}

print(id_to_char)

The encode function for character-level tokenization just returns a sequence of now encoded token IDs.

In [ ]:
def encode_without_unk(text):
    # Get token ID for each character in order and return as list
    return [char_to_id[char] for char in text]

In [ ]:
text = "Maybe it's the last night"

encoded = encode_without_unk(text)

print("Text:", text)
print("Encoded:", encoded)

The decode function takes the encoded sequence and converts it back into the characters.

In [ ]:
def decode(token_ids):
    return "".join( # join with no separator
        id_to_char[token_id] # corresponding character
        for token_id in token_ids
    )

In [ ]:
# Decode the encoded text
decoded = decode(encoded)

print("Original text:", text)
print("Decoded:", decoded)

It's impotant to notice that encoding only works with text sequences only have characters from the 56-character vocabulary.

In [ ]:
# Uncomment to try encoding text that contains unknown characters
# encode_without_unk("BADBADNOTGOOD")

We'll handle unknown characters by adding an unknown token to the vocabulary.

In [ ]:
# Unknown token
unk_token = "<UNK>"

# Add unknown token and assign ID
char_to_id[unk_token] = len(char_to_id)

# Update reverse vocabulary
id_to_char[char_to_id[unk_token]] = unk_token

print("UNK token ID:", char_to_id[unk_token])
print("Size of vocabulary:", len(char_to_id)) # should be +1 from before

In [ ]:
# Updated encode function
def encode_with_unk(text):
    unk_id = char_to_id["<UNK>"]

    return [
        char_to_id.get(char, unk_id) # get ID or default to unknown ID
        for char in text
    ]

Unknown tokens should be mapped to the unknown token ID, in this case 56.

In [ ]:
encoded = encode_with_unk("BADBADNOTGOOD")
all_unknowns = encode_with_unk("DDDDD")

print(encoded)
print(decode(encoded))
print()
print(all_unknowns)
print(decode(all_unknowns))

## Word-Level Tokenization

Conceptually almost the same, but now we separate tokens on with a delimiter, in this case whitespace for simplicity.

In [ ]:
words = corpus.split()
print(words)

The first limitation seen is that some split text includes punctuation like `"now,"`. Since this is deliberately for practice, we'll keep the splitting implementation simple.

In [ ]:
# Get every unique split in training corpus
word_vocabulary = sorted(set(words))

# Map each word to a unique token ID
word_to_id = {
    word: index # word:token ID pair
    for index, word in enumerate(word_vocabulary)
}

# Reverse vocabulary (decoding purposes)
id_to_word = {
    index: word
    for word, index in word_to_id.items()
}

In [ ]:
print(word_vocabulary)
print("Vocabulary size:", len(word_vocabulary))

print("\nWord -> Token ID mapping:", word_to_id)

print("\nToken ID -> Word mapping:", id_to_word)

We'll handle unknowns with a special unknown token again, then implement the encode/decode functions.

In [ ]:
# Unknown token
unk_token = "<UNK>"

# Append unknown token
word_to_id[unk_token] = len(word_to_id)
id_to_word[word_to_id[unk_token]] = unk_token

print("UNK token ID:", word_to_id[unk_token])
print("Size of vocabulary:", len(word_to_id))

In [ ]:
def encode_words(sequence):
    unk_id = word_to_id["<UNK>"]

    tokens = sequence.split() # split sequence

    return [
        word_to_id.get(token, unk_id) # get ID or default to unknown ID
        for token in tokens
    ]

def decode_words(token_seq):
    return " ".join(
        id_to_word[token_id] # corresponding word
        for token_id in token_seq
    )

In [ ]:
text = "Maybe it's the last night"
encoded = encode_words(text)

print("Text:", text)
print("Encoded:", encoded)
print("Decoded:", decode_words(encoded))

unk_text = "Childish Gambino"
encoded = encode_words(unk_text)

print("\nText w/ unknown words:", unk_text)
print("Encoded:", encoded)
print("Decoded:", decode_words(encoded))

We can now compare the character-level and word-level tokenization.

In [ ]:
text = "See now, I just wanna live forever, and ever"

encoded_chars = encode_with_unk(text)
encoded_words = encode_words(text)

# Print token counts
print("Character-level tokens:", len(encoded_chars))
print("Word-level tokens:", len(encoded_words))

## BPE

Now seeing how character/word tokenization works conceptually, the next natural progression is byte-pair encoding.

In [ ]:
# sample BPE corpus
bpe_corpus = "hug hugging hugger huge"

In [ ]:
# Split each word into individual characters
bpe_split = bpe_corpus.split()
bpe_words = [
    list(word)
    for word in bpe_split
]

print(bpe_words) # current vocabulary

We make a helper function that counts the frequency of each adjacent pair in each word.

In [ ]:
from collections import Counter

def count_pairs(words):
    pair_count = Counter() # dictionary of (char1, char2):count pairs

    for word in words:
        for i in range(len(word) - 1): # iterative through every char that has a following char
            pair = word[i], word[i + 1] # pair them
            pair_count[pair] += 1 # increment count for that pair

    return pair_count

We have a tie between `('h', 'u')` and `('u', 'g')` with count `4`, for learning purpose, we'll explicitly select the first pair to merge.

In [ ]:
pair_frequencies = count_pairs(bpe_words)

print(pair_frequencies)

In [ ]:
pair_to_merge = ("h", "u")

Using the chosen pair to merge, we define a function that recalculates the vocabulary by replacing the correponding adjacent pairs with the chosen pair that was merged.

In [ ]:
def merge_pair(words, pair):
    merged_words = [] # new corpus

    left, right = pair # unpack pair
    merged_token = left + right # first, second -> firstsecond

    for word in words:
        new_word = []
        i = 0

        while i < len(word): # iterative through each part of word

            # If next 2 parts match, add merged token
            if (
                i < len(word) - 1
                and word[i] == left
                and word[i + 1] == right
            ):
                new_word.append(merged_token)
                i += 2 # increment 2 to move over parts that were merged

            # Else just append and increment
            else:
                new_word.append(word[i])
                i += 1

        merged_words.append(new_word)

    return merged_words

In [ ]:
# New corpus with merged pairs
bpe_words = merge_pair(
    bpe_words,
    pair_to_merge
)

print(bpe_words)

# Recount pairs
pair_counts = count_pairs(bpe_words)

print(pair_counts)

That was a typical *cycle* of BPE, but we can automate the cycle until set threshold is met. Typically BPE stops merging when a vocabulary size or a set number of merges is reached.

In [ ]:
def train_bpe(corpus, merge_limit):
    # Split corpus into individual characters
    words_split = corpus.split()
    words = [
        list(word)
        for word in words_split
    ]

    # Merged pairs
    merges = []

    for _ in range(merge_limit):

        # Count pair frequences
        pair_frequencies = count_pairs(words)

        # Break loop if there's no pairs to merge
        if not pair_frequencies:
            break

        # Most frequent adjacent pair
        chosen_pair = pair_frequencies.most_common(1)[0][0] # from 1st index, get pair from pair:count

        # Save merged pair
        merges.append(chosen_pair)

        # Apply merge to corpus
        words = merge_pair(words, chosen_pair)

    return words, merges

In [ ]:
trained_words, merges = train_bpe(
    bpe_corpus,
    merge_limit=5
)

print("Final tokenized corpus:")
for word in trained_words:
    print(word)

print("\nLearned merges:")
for merge in merges:
    print(merge)

We can new add the new tokens from BPE to the initial vocabulary.

In [ ]:
def build_bpe_vocabulary(corpus, merges_to_append):
    # Initial character vocab
    vocab = set(
        char
        for word in corpus
        for char in word
    )

    # Add every token created by BPE
    for left, right in merges_to_append:
        vocab.add(left + right)

    return sorted(vocab)

In [ ]:
bpe_vocab = build_bpe_vocabulary(
    bpe_corpus,
    merges
)

print("BPE vocabulary:")
print(bpe_vocab)

print("Vocabulary size:", len(bpe_vocab))

Map BPE tokens to token IDs and reverse.

In [ ]:
# Encode
bpe_to_id = {
    token: index
    for index, token in enumerate(bpe_vocab)
}

# Decode
id_to_bpe = {
    index: token
    for token, index in bpe_to_id.items()
}

In [ ]:
# Add unknown token
unk_token = "<UNK>"

# Append unknown token
bpe_to_id[unk_token] = len(bpe_to_id)
id_to_bpe[bpe_to_id[unk_token]] = unk_token

print("UNK token ID:", bpe_to_id[unk_token])
print("Size of vocabulary:", len(bpe_to_id))

Now we make a function that takes a word and turns them into tokens.

In [ ]:
def encode_bpe_word(word, merges):
    tokens = [list(word)]

    # Apply all BPE merges to word
    for pair in merges:
        tokens = merge_pair(tokens, pair)

    return tokens[0] # return first

In [ ]:
print(encode_bpe_word("hugging", merges))
print(encode_bpe_word("hugger", merges))
print(encode_bpe_word("huge", merges))

print("\nEncoding unfamiliar text:", encode_bpe_word("huggingehugerhugen", merges))

The encode function takes in a word, uses the merge vocabulary and corresponding token IDs to encode it. Since it the BPE vocabulary has individual characters and subwords it merged, we are able to handle unfamililar words to a degree.

In [ ]:
def encode_bpe(word, merges, token_to_id):
    unk_id = token_to_id["<UNK>"]
    tokens = encode_bpe_word(word, merges)

    return [
        token_to_id.get(token, unk_id)
        for token in tokens
    ]

In [ ]:
unk_word = "huggingehugerhugen"

tokens = encode_bpe_word(unk_word, merges)
token_ids = encode_bpe(unk_word, merges, bpe_to_id)

print("Word:", unk_word)
print("BPE token split:", tokens)
print("Token IDs:", token_ids)

Decode function to complete the encode-decode interface the same as the character and word tokenizers.

In [ ]:
def decode_bpe(token_ids):
    tokens = [
        id_to_bpe[token_id] # corresponding token
        for token_id in token_ids
    ]

    return "".join(tokens) # join with no separator

In [ ]:
encoded = encode_bpe("hugging", merges, bpe_to_id)
decoded = decode_bpe(encoded)

print("Encoded:", encoded)
print("Decoded:", decoded)

### Use On Original Corpus

We'll go about applying BPE to the original corpus, in order to compare it to the character and word tokenizers.

In [ ]:
corpus_words = sorted(set(corpus.split()))
corpus_char_vocab = sorted(set(corpus))

print("Word vocabulary size:", len(corpus_words))
print("Character vocabulary size:", len(corpus_char_vocab))

We'll aim for a subword vocabulary with size **56 < size < 208**.

In [ ]:
corpus_words, merges = train_bpe(corpus, merge_limit=50)

print("Final tokenized corpus:", corpus_words)
print("Learned merges:", merges)
print("\nNumber of learned merges:", len(merges)) # check correct number of merges

In [ ]:
bpe_vocab = build_bpe_vocabulary(
    corpus,
    merges
)

print("BPE vocabulary:")
print(bpe_vocab)

print("\nVocabulary size:", len(bpe_vocab))

In [ ]:
# Encode
bpe_to_id = {
    token: index
    for index, token in enumerate(bpe_vocab)
}

# Decode
id_to_bpe = {
    index: token
    for token, index in bpe_to_id.items()
}

In [ ]:
# Add unknown token
unk_token = "<UNK>"

# Append unknown token
bpe_to_id[unk_token] = len(bpe_to_id)
id_to_bpe[bpe_to_id[unk_token]] = unk_token

print("UNK token ID:", bpe_to_id[unk_token])
print("Size of vocabulary:", len(bpe_to_id))

In [ ]:
sequence = "Maybe it's the last night"

tokens = encode_bpe_word(sequence, merges)

encoded = encode_bpe(sequence, merges, bpe_to_id)
decoded = decode_bpe(encoded)

print("Sequence:", sequence)
print("BPE token split:", tokens)

print("\nToken IDs:", encoded)
print("Decoded:", decoded)


## Comparison

We can see firsthand how character-level tokenization tokenizes into the longest sequence, word-level tokenization has the least, and BPE stays in between.

The important thing to remember about BPE isn't that it lands numerically in the middle of the other two tokenizers, but that it tries to get the benefits of both sides.

In [ ]:
sample = "Maybe it's the last night"

char_tokens = list(sample) # split into single characters

word_tokens = sample.split() # split words

bpe_tokens = encode_bpe_word(
    sample,
    merges
)

print("Character tokens:", char_tokens)
print("Character count:", len(char_tokens))

print("\nWord tokens:", word_tokens)
print("Word count:", len(word_tokens))

print("\nBPE tokens:", bpe_tokens)
print("BPE count:", len(bpe_tokens))